# Day 2 Starter Kit — Campus Operations Grounded Agent

**Your task:** build an agent that answers questions about this campus-operations dataset —
and, critically, **never states a connection between two records that don't actually
reference the same event.**

This notebook gives you working retrieval + a baseline RAG pipeline (the *broken* version,
on purpose — you saw why in the Day 1 demo). Your job is the section marked **YOUR TASK**:
add a grounding mechanism so the agent stops fabricating connections.

**Grounding approach: use Neo4j.** Day 1's fix (notebook 3) used a Neo4j graph, and you'll
each create your own free Neo4j Aura account for Day 2 — so build your grounding mechanism
the same way, not a different one, and judging stays consistent with what was taught. The
setup section below already has the Neo4j connection boilerplate wired up, matching notebook
3 exactly, so you can go straight to designing your graph schema and Cypher checks instead of
rebuilding that plumbing from scratch.

Framework is your choice for everything else — keep building in this notebook, or rebuild the
same idea in n8n (see the accompanying n8n starter guide, which also uses a Neo4j node for the
grounding check). Either is fine; judging doesn't care which you used, as long as the grounding
mechanism itself is graph-based.

## 1. Setup

In [ ]:
%pip install -q sentence-transformers hnswlib openai python-dotenv neo4j

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import hnswlib
import os
from openai import OpenAI

def get_secret(name):
    """Reads a secret from Colab's Secrets manager (padlock icon in the left
    sidebar) if running on Colab; falls back to a local .env file otherwise."""
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv(name)

model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded OK")

In [ ]:
# Add OPENAI_API_KEY to Colab's Secrets manager (padlock icon), or a local .env file.
client = OpenAI(api_key=get_secret("OPENAI_API_KEY"))
print("Client ready")

### Connect to Neo4j Aura

Create a free [Neo4j Aura](https://neo4j.com/cloud/aura-free/) instance if you haven't
already, then add `NEO4J_URI`, `NEO4J_USER`, and `NEO4J_PASSWORD` to Colab's Secrets manager.
Same connection pattern as Day 1's notebook 3 — if you hit `ServiceUnavailable`, the instance
auto-paused from inactivity; re-run this cell, or `driver.close()` + recreate.

In [ ]:
from neo4j import GraphDatabase

uri = get_secret("NEO4J_URI")
user = get_secret("NEO4J_USER")
password = get_secret("NEO4J_PASSWORD")

driver = GraphDatabase.driver(uri, auth=(user, password))

def run_write(query, **params):
    with driver.session() as session:
        session.run(query, **params)

def run_read(query, **params):
    with driver.session() as session:
        result = session.run(query, **params)
        return [record.data() for record in result]

print(run_read("RETURN 'Connection successful' AS message")[0]["message"])

## 2. The dataset — campus operations & student services

14 categories, 172 records. Four pairs are deliberately overlapping in vocabulary: **Fees_Finance <-> Scholarships** (both involve money), **Hostel_Accommodation <-> Facilities_Maintenance** (both mention "room"), **Transport <-> Parking_Permits** (both mention "pass"/"permit"), and **Events_Clubs <-> Alumni_Relations** (both mention "event"/"funding"). Same trap pattern from Day 1 — just a new domain, and more of it.

**Some records name a specific person or ID — most don't.** Look closely and you'll notice some sentences say things like `"Student S001's tuition fee payment..."` or reference a ticket code like `HMT-58`. Where two records name the *same* ID across categories, that's a real connection your graph should be able to affirm. Where the ID differs (or there's no name at all — most records), it's a vocabulary trap: similar-sounding, but not the same event. Your grounding check needs to tell these apart — not by category, but by whether the data actually shares an identifiable entity.

In [ ]:
campus_categories = {
    "Admissions": [
        "A new student's admission application was reviewed by the committee",
        "Entrance exam scorecard was verified for a incoming student",
        "Admission offer letter was issued to a selected candidate",
        "A student's transfer application from another college was processed",
        "Document verification was completed for a first-year admission",
        "Admission committee flagged a discrepancy in a submitted transcript",
        "Waitlisted candidate was offered a seat after a dropout",
        "Provisional admission was confirmed pending final marksheet submission",
        "International student's visa documents were reviewed for admission",
        "Admission counselor scheduled a call with a prospective student",
        "Student S013's admission document verification was completed ahead of orientation",
    ],
    "Fees_Finance": [
        "Tuition fee payment of 45000 rupees was received from a student",
        "A late fee penalty was applied to an overdue tuition payment",
        "Fee receipt was generated for the current semester payment",
        "A student requested an installment plan for pending tuition fees",
        "Refund was processed for a cancelled admission's fee payment",
        "Finance office reconciled the semester's total fee collection",
        "A bounced fee payment was flagged for reprocessing",
        "Hostel fee and tuition fee were combined into one invoice",
        "A student's fee payment was adjusted after a course change",
        "Finance team sent a reminder for an unpaid tuition balance",
        "Student S001's tuition fee payment of 42000 rupees was received for the fall semester",
        "Student S002 was charged a late fee penalty on last month's tuition invoice",
        "Student S004's tuition installment plan was approved by the finance office",
    ],
    "Scholarships": [
        "Scholarship disbursement of 15000 rupees was processed for a first-year student",
        "A merit scholarship application was approved by the review panel",
        "Scholarship renewal was submitted for the upcoming academic year",
        "A need-based scholarship award was credited to a student's account",
        "Scholarship committee rejected an incomplete application",
        "Sports scholarship was granted to a state-level athlete",
        "A student's scholarship was revoked due to low attendance",
        "External scholarship provider confirmed funding for a student",
        "Scholarship disbursement was delayed pending document resubmission",
        "A minority scholarship application was forwarded for government approval",
        "Student S001's merit scholarship of 20000 rupees was disbursed for the fall semester",
        "Student S002's scholarship renewal was approved for the second year",
        "Student S003's sports scholarship was confirmed by the athletics department",
    ],
    "Hostel_Accommodation": [
        "Room 214 was allotted to a new student this week",
        "A hostel room change request was approved for a second-year student",
        "Hostel warden resolved a roommate conflict in Block C",
        "A student checked out of the hostel at the end of the semester",
        "New hostel admission form was submitted for the upcoming term",
        "Hostel curfew violation was reported to the warden's office",
        "A student requested a single-occupancy room upgrade",
        "Hostel mess registration was completed for a new resident",
        "Room allotment list was published for incoming hostel students",
        "A student vacated their room early due to a course withdrawal",
        "Student S006's room change request to Room 118 was approved",
        "Student S007 reported a curfew violation warning from the hostel warden",
        "A hostel maintenance ticket (HMT-58) was filed by the student in Room 310 for a broken window",
    ],
    "Facilities_Maintenance": [
        "A maintenance fee payment of 2000 rupees was recorded for Block C rooms",
        "Electrician was dispatched to fix a wiring issue in the hostel block",
        "Plumbing repair request was logged for a leaking bathroom",
        "Facilities team repainted the corridor walls in the academic block",
        "Air conditioning unit was serviced in the seminar hall",
        "A broken window pane was replaced in the library building",
        "Elevator maintenance was scheduled for the administrative block",
        "Facilities flagged a structural crack for engineering inspection",
        "Water cooler was repaired in the student common area",
        "Furniture replacement was approved for the outdated classroom desks",
        "Facilities resolved maintenance ticket HMT-58, replacing the broken window in Room 310",
        "Facilities serviced the air conditioning unit in Room 118 at Student S006's request",
        "Student S008 filed a complaint about a leaking bathroom in a different block",
        "Student S012 reported a broken door lock in Room 118 during the following semester",
    ],
    "Library": [
        "A student issued three reference books for the semester",
        "Overdue library fine was paid by a student for a late return",
        "New journal subscriptions were added to the digital library",
        "A damaged book was reported and replaced in the collection",
        "Library extended its hours during the examination period",
        "A student requested an inter-library loan for a rare textbook",
        "Library staff reshelved returned books after the rush week",
        "E-book access was granted to a student for an online course",
        "A lost book fee was charged to a student's account",
        "Library conducted an annual stock verification of its collection",
        "Student S013 issued two reference textbooks for their first-year courses",
        "Student S015 was charged a lost book fee after misplacing a library textbook",
    ],
    "IT_Helpdesk": [
        "A student's login credentials were reset after a lockout",
        "Wi-Fi connectivity issue was reported in the academic block",
        "IT helpdesk resolved a printer malfunction in the computer lab",
        "A student's email account was migrated to the new domain",
        "Software installation request was completed for a lab machine",
        "IT flagged a phishing email reported by a faculty member",
        "A student's laptop was unable to connect to the campus network",
        "IT helpdesk ticket was closed after a successful password reset",
        "Server downtime affected the online portal for two hours",
        "A student requested VPN access for remote library resources",
        "Student S014's laptop was unable to connect to the campus Wi-Fi network",
    ],
    "Placement_Careers": [
        "A student was shortlisted for a summer internship interview",
        "Placement cell scheduled a pre-placement talk with a recruiter",
        "A student received a job offer through the campus placement drive",
        "Resume workshop was conducted for final-year students",
        "Placement officer followed up with a company on pending offers",
        "A student withdrew from the placement process after a personal offer",
        "Mock interview session was organized for placement preparation",
        "Company representative visited campus for a recruitment drive",
        "A student's internship certificate was verified by the placement office",
        "Placement statistics were compiled for the annual report",
        "Student S013 was shortlisted for a summer internship interview by a recruiting company",
    ],
    "Events_Clubs": [
        "The robotics club organized a workshop for new members",
        "Cultural fest committee finalized the event schedule",
        "A student club requested funding for an upcoming competition",
        "Annual sports day events were scheduled across three days",
        "The photography club held an exhibition in the main hall",
        "Student council approved a new club's registration",
        "A guest lecture was organized by the entrepreneurship cell",
        "Debate club members prepared for an inter-college competition",
        "The music club performed at the college's annual day",
        "A club's event proposal was rejected due to a venue conflict",
        "The robotics club's workshop for new members was led by its club president (Club RC-1)",
        "The robotics club's president also represented the club (Club RC-1) at the inter-college competition",
    ],
    "Transport": [
        "A new bus route was added for students from the north campus",
        "Transport pass renewal was completed for the semester",
        "A student reported a delay on the college shuttle service",
        "Transport office adjusted the bus schedule for exam week",
        "A damaged bus seat was reported to the transport department",
        "Student transport fee was included in the semester invoice",
        "A new driver was assigned to the evening shuttle route",
        "Transport helpdesk resolved a lost bus pass complaint",
        "Additional buses were arranged for the college fest weekend",
        "A student requested a refund for an unused transport pass",
        "Student S009's bus pass was renewed for the spring semester",
        "Student S011 filed a complaint about a delayed shuttle on the north campus route",
    ],
    "Parking_Permits": [
        "A student's parking permit application was approved for the fall semester",
        "Parking enforcement issued a warning for an expired permit",
        "A visitor parking pass was requested for a campus event",
        "Parking permit renewal was completed for a returning student",
        "A student reported a lost parking permit sticker",
        "Parking office resolved a dispute over an assigned spot",
        "A faculty parking permit was upgraded to a reserved spot",
        "Parking permit fee was waived for a shared carpool application",
        "A damaged permit sticker was replaced at the parking office",
        "Student S016's parking permit application was approved for the spring term",
        "Parking violation notice was issued for an unregistered vehicle",
        "Student S009's parking permit was renewed for the spring semester",
        "Student S010 received a parking violation notice for an unregistered vehicle",
    ],
    "Alumni_Relations": [
        "An alumni donation was received for the new library wing",
        "Alumni association organized a networking event for graduates",
        "A former student's contact details were updated in the alumni database",
        "Alumni newsletter was sent out for the quarterly update",
        "An alumni-funded scholarship endowment was reviewed by the committee",
        "Alumni relations office confirmed a guest speaker for homecoming",
        "A batch reunion event was scheduled for December",
        "Alumni association approved funding for a new mentorship program",
        "A distinguished alumnus was invited to the annual convocation",
        "Alumni donation records were reconciled for the fiscal year",
        "Alumnus A001's donation for the new library wing was formally acknowledged by the college",
        "Alumnus A001 attended the homecoming networking event hosted by the alumni association",
        "Alumnus A002's mentorship program funding was approved by the alumni association",
        "Alumnus A002 was invited to speak at the annual convocation as a distinguished alumnus",
    ],
    "Exam_Results": [
        "A student's semester grade report was published on the portal",
        "Exam committee resolved a discrepancy in a marksheet",
        "A student requested a re-evaluation of their exam paper",
        "Final exam results were released for the engineering department",
        "A student's backlog exam was scheduled for the next session",
        "Exam office issued a duplicate marksheet for a lost original",
        "A grade correction was approved after a clerical error was found",
        "Semester GPA was recalculated after a course withdrawal",
        "Exam hall ticket was issued for the upcoming semester exams",
        "A student's provisional degree certificate was requested after final results",
        "Student S014's backlog exam for a failed course was scheduled for next session",
    ],
    "Campus_Security": [
        "Campus security responded to a report of a stolen bicycle",
        "A lost ID card was recovered and returned through campus security",
        "Security patrol flagged an unlocked door in the academic block",
        "A student reported a suspicious person near the hostel gate",
        "Campus security reviewed CCTV footage after a minor incident",
        "A found wallet was logged at the campus security office",
        "Security escorted a student to their vehicle after a late-night event",
        "A broken security gate was reported for repair",
        "Campus security issued a visitor pass for an external vendor",
        "An access card malfunction was resolved by the security office",
        "Student S015 reported a lost ID card to campus security",
        "Student S016's vehicle was found unlocked by a security patrol and secured",
    ],
}



campus_texts = []
campus_labels = []
for category, sentences in campus_categories.items():
    for s in sentences:
        campus_texts.append(s)
        campus_labels.append(category)

print(f"Total records: {len(campus_texts)}")
print(f"Categories: {len(campus_categories)}")

## 3. Build the retrieval index (given — this part works)

In [ ]:
campus_embeddings = model.encode(campus_texts, show_progress_bar=True)
campus_embeddings = np.ascontiguousarray(campus_embeddings.astype('float32'))

dim = campus_embeddings.shape[1]
campus_index = hnswlib.Index(space='cosine', dim=dim)
campus_index.init_index(max_elements=len(campus_texts), ef_construction=200, M=16)
campus_index.add_items(campus_embeddings, np.arange(len(campus_texts)))
campus_index.set_ef(50)
print(f"Index built: {len(campus_texts)} records")

In [ ]:
def retrieve(question, k=3):
    q_emb = model.encode([question])[0].astype('float32').reshape(1, -1)
    labels, distances = campus_index.knn_query(q_emb, k=k)
    return [(int(idx), campus_texts[idx], campus_labels[idx]) for idx in labels[0]]

# quick sanity check
for idx, text, label in retrieve("Tell me about a hostel room issue", k=3):
    print(f"[{label}] {text}")

## 4. Baseline RAG (given — this is the *broken* version)

This retrieves and generates, exactly like your Week 2 pipeline. It will work fine on straightforward questions and **fail on the adversarial ones** — that failure is the whole point of the exercise, and it's what you're fixing in the next section.

In [ ]:
def baseline_rag_answer(question, k=3):
    results = retrieve(question, k=k)
    context = "\n".join(f"- {text}" for _, text, _ in results)
    prompt = f'''Answer the question using ONLY the context below. If the context doesn't contain the answer, say so.

Context:
{context}

Question: {question}

Answer:'''
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )
    return {"question": question, "retrieved": results, "answer": response.choices[0].message.content}

# try it on an easy question
result = baseline_rag_answer("What happened with room 214?")
print(result["answer"])

## 5. YOUR TASK — build the graph and the grounding check

The baseline above has no way of knowing whether two retrieved records actually describe the
same event — it just hands the LLM whatever's semantically similar and trusts it to reason
correctly. It won't, reliably (you saw this on Day 1).

**Build a `grounded_answer(question, k=3)` function that fixes this, using the Neo4j
connection above.** Two things to design:

1. **A graph schema for (at least some of) this dataset.** You don't need to model all 142
   records — notebook 3 only modeled two small, specific chains, not its whole corpus. Pick the
   records involved in the categories you think are riskiest (the four overlapping pairs above
   are a good place to start), and represent them as nodes/relationships — e.g. an `:Event` node
   per record, linked to the real entities it involves (a student, a room, a block, a club).
   Link entities across records **only where the data actually supports it** (a shared ID,
   ticket number, or name) — not by category, and not by guesswork either way.
2. **A Cypher check in `grounded_answer`** — before answering, check whether the retrieved
   records correspond to graph nodes, and if so, whether a path actually connects them
   (`shortestPath`, like notebook 3's `connected_order` check). State the graph's answer to the
   LLM explicitly either way: if no path exists, say so; if one genuinely does, say that too —
   don't let a "be cautious" instruction turn into always denying any connection.

A prompt instruction alone ("don't connect unrelated records") is **not** enough — this was
directly tested on Day 1 and shown to fail on its own. The graph check is what has to do the
real work; the prompt just has to state the graph's verdict plainly, whichever way it goes.

The rubric scores whether your graph check actually stops fabricated connections **and**
correctly affirms a real one when tested — not how much of the dataset you modeled. A small,
correct graph beats a large, unused one, and a graph that only ever says "unrelated" isn't
actually checking anything.

In [ ]:
def grounded_answer(question, k=3):
    # TODO: implement this.
    # Suggested skeleton:
    #   1. retrieve(question, k) same as before
    #   2. for the retrieved records, check whether they map to nodes in your graph
    #   3. if they do, query Neo4j for whether a path connects them (see notebook 3's
    #      connected_order / shortestPath pattern)
    #   4. build the prompt so the graph's verdict (connected / not connected) is stated
    #      explicitly, not left for the LLM to infer from wording
    raise NotImplementedError("Build your grounding logic here")

## 6. Self-test (public — separate from the judges' hidden test set)

These are yours to test with. The judges will run a **different**, held-out set of adversarial questions (6 of them, across all four overlapping pairs plus two harder cross-category cases) at judging time — passing these doesn't guarantee passing those, but if your agent fails these, it will fail those too.

In [ ]:
self_test_questions = [
    "Did the student who got a new hostel room also get billed for a repair?",
    "Was a scholarship used to pay someone's tuition fee this semester?",
    "Is the library fine related to the IT helpdesk ticket?",
]

for q in self_test_questions:
    print("Q:", q)
    print("Baseline (should likely fail):", baseline_rag_answer(q)["answer"][:200])
    print("---")
    # once grounded_answer is implemented:
    # print("Grounded:", grounded_answer(q)["answer"][:200])
    print()

## 7. Export embeddings (optional — only needed if you're building the n8n version instead of/alongside this notebook)

Run this once to produce a JSON file the n8n starter guide uses to seed its retrieval step, so you don't have to recompute embeddings inside n8n.

In [ ]:
import json

export = {
    "texts": campus_texts,
    "labels": campus_labels,
    "embeddings": campus_embeddings.tolist(),
}
with open("campus_embeddings_export.json", "w") as f:
    json.dump(export, f)

print("Exported campus_embeddings_export.json —", len(campus_texts), "records")

## 8. Submission

Submit this notebook (with `grounded_answer` implemented) — or your equivalent n8n workflow export — by the deadline announced on Day 1. Make sure it runs top-to-bottom on a fresh runtime before you submit; a notebook that only works because of leftover session state will fail the judges' fresh run.